STAGE 1 IMPLEMENTATION (Corpus + Extraction)

In [71]:
!pip install pdfplumber pymupdf

In [3]:
import pdfplumber
import os

In [4]:
#load pdf
pdf_path = "../data/raw/force_laws_motion.pdf"

In [6]:
#extract text from pdf
all_text = ""

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            all_text += f"\n\n--- PAGE {i+1} ---\n\n"
            all_text += text

print("Extraction done")

Extraction done


In [7]:
print(all_text[:2000])



--- PAGE 1 ---

8
C
hapter
FFFFF LLLLL MMMMM
OOOOORRRRRCCCCCEEEEE AAAAANNNNNDDDDD AAAAAWWWWWSSSSS OOOOOFFFFF OOOOOTTTTTIIIIIOOOOONNNNN
In the previous chapter, we described the In our everyday life we observe that some
motion of an object along a straight line in effort is required to put a stationary object
terms of its position, velocity and acceleration. into motion or to stop a moving object. We
We saw that such a motion can be uniform ordinarily experience this as a muscular effort
or non-uniform. We have not yet discovered and say that we must push or hit or pull on
what causes the motion. Why does the speed an object to change its state of motion. The
of an object change with time? Do all motions concept of force is based on this push, hit or
require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What
this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt a
attempt to quench all such curiosities. force. Howe

In [8]:
#cleaning text for better processing
def advanced_clean_text(text):
    import re
    
    # Remove "Reprint" noise
    text = re.sub(r'Reprint \d{4}-\d{2}', ' ', text)
    
    # Remove standalone numbers (like page numbers)
    text = re.sub(r'\b\d+\b', ' ', text)
    
    # Remove empty brackets
    text = re.sub(r'\(\s*\)', ' ', text)
    # Remove page markers
    text = re.sub(r'--- PAGE \d+ ---', ' ', text)
    
    # Remove repeated uppercase noise (e.g. OOOOORRRRR)
    text = re.sub(r'\b[A-Z]{3,}\b', ' ', text)
    
    # Fix broken words like "C hapter"
    text = re.sub(r'\b([A-Za-z])\s+([a-z])\b', r'\1\2', text)
    
    # Remove figure references
    text = re.sub(r'Fig\.\s*\d+(\.\d+)?', ' ', text)
    
    # Remove (a), (b), etc.
    text = re.sub(r'\([a-z]\)', ' ', text)
    
    # Fix sentence spacing (add period space)
    text = re.sub(r'\.\s+', '. ', text)
    
    # Replace newlines with space
    text = re.sub(r'\n+', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [9]:
#clened text not ideal but acceptable for now. We can always improve this later.
cleaned_text = advanced_clean_text(all_text)

print(cleaned_text[:2000])

--- --- C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the speed an object to change its state of motion. The of an object change with time? Do all motions concept of force is based on this push, hit or require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt a attempt to quench all such curiosities. force. However, we always see or feel the effect For many centuries, the problem of of a force. It can only be explained by motion 

In [10]:
clean_path = "../data/processed/cleaned_text.txt"

with open(clean_path, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print("Cleaned text saved")

Cleaned text saved


STAGE 1 — PART 2: STRUCTURE DETECTION

In [11]:
#Split into smaller units
def split_into_blocks(text, max_len=300):
    words = text.split()
    
    blocks = []
    current = []
    
    for word in words:
        current.append(word)
        
        if len(current) >= max_len:
            blocks.append(" ".join(current))
            current = []
    
    if current:
        blocks.append(" ".join(current))
    
    return blocks

blocks = split_into_blocks(cleaned_text)

print(len(blocks))
print(blocks[0][:500])

21
--- --- C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the sp


In [12]:
#Detect types (core logic)
#concept
#example
#question
def classify_block(text):
    t = text.lower()
    
    # Example detection (stronger)
    if "example" in t:
        return "example"
    
    # Question detection (better rule)
    if "?" in text and len(text) < 200:
        return "question"
    
    return "concept"

In [13]:
#Apply classification
structured_blocks = []

for block in blocks:
    structured_blocks.append({
        "text": block,
        "type": classify_block(block)
    })

structured_blocks[:5]

[{'text': '--- --- C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the speed an object to change its state of motion. The of an object change with time? Do all motions concept of force is based on this push, hit or require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt a attempt to quench all such curiosities. force. However, we always see or feel the effect For many centuries, the problem of of a force. It can only be explained

In [14]:

#Inspect classification
for item in structured_blocks[:20]:
    print(item)
    print("------")

{'text': '--- --- C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the speed an object to change its state of motion. The of an object change with time? Do all motions concept of force is based on this push, hit or require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt a attempt to quench all such curiosities. force. However, we always see or feel the effect For many centuries, the problem of of a force. It can only be explained 

Key insight (this is important)

👉 we have reached:

✔️ “usable raw structure”
❌ “not retrieval-ready yet”

STEP 3 — CHUNKING WITH OVERLAP

In [87]:
!pip install transformers

In [2]:
#load tokenizer
from transformers import AutoTokenizer

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


side quest(comparision between different tokenizer)

In [15]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")

c:\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\himkar vashistha\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to 

In [16]:
sample_text = """
Newton's first law of motion states that a body remains at rest
or moves with constant velocity unless acted upon by an external force.
A force of 12.75 N acts on a 5 kg object.
"""

In [17]:
bert_tokens = bert_tokenizer.tokenize(sample_text)
gpt2_tokens = gpt2_tokenizer.tokenize(sample_text)
t5_tokens = t5_tokenizer.tokenize(sample_text)

In [18]:
print("BERT Tokens:", len(bert_tokens))
print(bert_tokens)

print("\nGPT2 Tokens:", len(gpt2_tokens))
print(gpt2_tokens)

print("\nT5 Tokens:", len(t5_tokens))
print(t5_tokens)

BERT Tokens: 41
['newton', "'", 's', 'first', 'law', 'of', 'motion', 'states', 'that', 'a', 'body', 'remains', 'at', 'rest', 'or', 'moves', 'with', 'constant', 'velocity', 'unless', 'acted', 'upon', 'by', 'an', 'external', 'force', '.', 'a', 'force', 'of', '12', '.', '75', 'n', 'acts', 'on', 'a', '5', 'kg', 'object', '.']

GPT2 Tokens: 45
['Ċ', 'New', 'ton', "'s", 'Ġfirst', 'Ġlaw', 'Ġof', 'Ġmotion', 'Ġstates', 'Ġthat', 'Ġa', 'Ġbody', 'Ġremains', 'Ġat', 'Ġrest', 'Ċ', 'or', 'Ġmoves', 'Ġwith', 'Ġconstant', 'Ġvelocity', 'Ġunless', 'Ġacted', 'Ġupon', 'Ġby', 'Ġan', 'Ġexternal', 'Ġforce', '.', 'Ċ', 'A', 'Ġforce', 'Ġof', 'Ġ12', '.', '75', 'ĠN', 'Ġacts', 'Ġon', 'Ġa', 'Ġ5', 'Ġkg', 'Ġobject', '.', 'Ċ']

T5 Tokens: 44
['▁Newton', "'", 's', '▁first', '▁law', '▁of', '▁motion', '▁states', '▁that', '▁', 'a', '▁body', '▁remains', '▁at', '▁rest', '▁or', '▁moves', '▁with', '▁constant', '▁velocity', '▁', 'unless', '▁', 'acted', '▁upon', '▁by', '▁an', '▁external', '▁force', '.', '▁A', '▁force', '▁of', '▁12

Tokenizer	Tokens	Strength
BERT	    41	    Best retrieval
GPT-2	    45	    Compact
T5	        44	    Flexible

we choose wordpiece bert tokenizer

In [19]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [20]:
#creating chunks instead of words we can also do it based on tokens which would be more accurate for our use case.
"""
def create_chunks(text, chunk_size=350, overlap=60):
    words = text.split()
    
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        
        chunk = " ".join(chunk_words)
        chunks.append(chunk)
        
        start = end - overlap
    
    return chunks"""
def create_token_chunks(text, chunk_size=350, overlap=60):

    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0

    while start < len(tokens):

        end = start + chunk_size

        chunk_tokens = tokens[start:end]

        chunk_text = tokenizer.decode(chunk_tokens)

        chunks.append(chunk_text)

        start = end - overlap

    return chunks

In [21]:
'''final_chunks = create_chunks(cleaned_text)

print(len(final_chunks))
print(final_chunks[0][:500])'''

final_chunks = create_token_chunks(cleaned_text)

print("Total chunks:", len(final_chunks))
print(final_chunks[0][:500])

Token indices sequence length is longer than the specified maximum sequence length for this model (7498 > 512). Running this sequence through the model will result in indexing errors


Total chunks: 26
- - - - - - c hapter in the previous chapter, we described the in our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. we we saw that such a motion can be uniform ordinarily experience this as a muscular effort or non - uniform. we have not yet discovered and say that we must push or hit or pull on what causes the motion. why does 


In [22]:
#adding meta data
'''import json

chunk_data = []

for i, chunk in enumerate(final_chunks):
    chunk_data.append({
        "id": i,
        "text": chunk,
        "chapter": "Force and Laws of Motion"
    })

with open("../data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunk_data, f, indent=4)

print("Chunks saved")'''

chunk_data = []

for i, chunk in enumerate(final_chunks):

    chunk_data.append({
        "id": i,
        "text": chunk,
        "chapter": "Force and Laws of Motion",
        "token_count": len(tokenizer.encode(chunk))
    })

chunk_data[:2]

[{'id': 0,
  'text': '- - - - - - c hapter in the previous chapter, we described the in our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. we we saw that such a motion can be uniform ordinarily experience this as a muscular effort or non - uniform. we have not yet discovered and say that we must push or hit or pull on what causes the motion. why does the speed an object to change its state of motion. the of an object change with time? do all motions concept of force is based on this push, hit or require a cause? if so, what is the nature of pull. let us now ponder about a ‘ force ’. what this cause? in this chapter we shall make an is it? in fact, no one has seen, tasted or felt a attempt to quench all such curiosities. force. however, we always see or feel the effect for many centuries, the problem of of a force. it ca

In [23]:
import json

with open("../data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunk_data, f, indent=4)

print("Token chunks saved")

Token chunks saved


In [24]:
print(chunk_data[0]["text"])

- - - - - - c hapter in the previous chapter, we described the in our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. we we saw that such a motion can be uniform ordinarily experience this as a muscular effort or non - uniform. we have not yet discovered and say that we must push or hit or pull on what causes the motion. why does the speed an object to change its state of motion. the of an object change with time? do all motions concept of force is based on this push, hit or require a cause? if so, what is the nature of pull. let us now ponder about a ‘ force ’. what this cause? in this chapter we shall make an is it? in fact, no one has seen, tasted or felt a attempt to quench all such curiosities. force. however, we always see or feel the effect for many centuries, the problem of of a force. it can only be explained by